# 09_Doo — Logistic Regression 하이퍼파라미터 튜닝

## 실험 원칙
Train 내부 5-fold 교차검증으로 튜닝하고 Validation에서 최종 비교한다. Test 데이터는 사용하지 않는다. 팀 공통 시드는 `random_state=42`다.

## 1. 데이터 확인
팀 전처리 결과를 사용한다. `churn=1`은 고객 이탈이다. 전처리에서 스케일링된 Train/Validation 데이터를 불러온다.

In [1]:
from pathlib import Path
import numpy as np, pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score, roc_auc_score

from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "data" / "preprocessed").exists():
    if ROOT.parent == ROOT:
        raise FileNotFoundError("data/preprocessed 폴더를 찾을 수 없음")
    ROOT = ROOT.parent

DATA_DIR = ROOT / "data" / "preprocessed"

X_train = pd.read_csv(DATA_DIR / 'X_train.csv')
y_train = pd.read_csv(DATA_DIR / 'y_train.csv')['churn']
X_val = pd.read_csv(DATA_DIR / 'X_val.csv')
y_val = pd.read_csv(DATA_DIR / 'y_val.csv')['churn']
print('X_train:', X_train.shape, '| X_val:', X_val.shape)


def evaluate(model, X, y):
    predictions = model.predict(X)
    probabilities = model.predict_proba(X)[:, 1]
    return {
        'accuracy': accuracy_score(y, predictions),
        'recall': recall_score(y, predictions),
        'precision': precision_score(y, predictions, zero_division=0),
        'f1': f1_score(y, predictions, zero_division=0),
        'roc_auc': roc_auc_score(y, probabilities),
    }


X_train: (2592, 10) | X_val: (864, 10)


## 2. 베이스라인 모델
L2 정규화 Logistic Regression을 기준선으로 학습한다. `class_weight='balanced'`는 이탈 클래스 탐지를 고려한다.

## 3. 하이퍼파라미터 튜닝
정규화 강도 `C`를 Train 내부 5-fold Stratified CV에서 ROC-AUC 기준으로 탐색한다. `C`가 작을수록 정규화가 강하다.

In [2]:
baseline = LogisticRegression(
    C=1.0, solver='liblinear', class_weight='balanced',
    max_iter=3000, random_state=42,
)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
search = RandomizedSearchCV(
    LogisticRegression(solver='liblinear', class_weight='balanced',
                       max_iter=3000, random_state=42),
    {'C': np.logspace(-3, 2, 30)}, n_iter=20, scoring='roc_auc',
    cv=cv, random_state=42, n_jobs=1, return_train_score=True,
)
search.fit(X_train, y_train)
tuned = search.best_estimator_
print('Best CV ROC-AUC:', round(search.best_score_, 4))
print('Best parameters:', search.best_params_)
baseline.fit(X_train, y_train)
rows = []
for name, model in [('Baseline', baseline), ('Tuned', tuned)]:
    train_metrics = evaluate(model, X_train, y_train)
    val_metrics = evaluate(model, X_val, y_val)
    rows.append({
        'model': name, 'train_accuracy': train_metrics['accuracy'],
        'val_accuracy': val_metrics['accuracy'],
        'recall': val_metrics['recall'], 'precision': val_metrics['precision'],
        'f1': val_metrics['f1'], 'roc_auc': val_metrics['roc_auc'],
        'train_val_gap': train_metrics['accuracy'] - val_metrics['accuracy'],
    })
comparison = pd.DataFrame(rows).set_index('model')
display(comparison.round(3))


Best CV ROC-AUC: 0.7639
Best parameters: {'C': np.float64(0.007278953843983154)}


,train_accuracy,val_accuracy,recall,precision,f1,roc_auc,train_val_gap
model,,,,,,,
Baseline,0.702,0.705,0.684,0.709,0.696,0.759,-0.003
Tuned,0.702,0.704,0.672,0.712,0.692,0.761,-0.002


## 4. 임계값 비교
기본 임계값 0.50 외에 여러 임계값을 비교한다. Recall 0.80 이상 후보 중 F1-score가 가장 높은 값을 선택한다.

In [3]:
tuned_proba = tuned.predict_proba(X_val)[:, 1];
threshold_rows = []
for threshold in np.arange(0.25, 0.71, 0.05):
    pred = (tuned_proba >= threshold).astype(int)
    threshold_rows.append({'threshold': threshold, 'recall': recall_score(y_val, pred),
                           'precision': precision_score(y_val, pred, zero_division=0),
                           'f1': f1_score(y_val, pred, zero_division=0), 'predicted_churn_count': int(pred.sum())})
threshold_df = pd.DataFrame(threshold_rows);
display(threshold_df.round(3))
candidates = threshold_df[threshold_df['recall'] >= 0.80]
selected = candidates.loc[candidates['f1'].idxmax()] if not candidates.empty else threshold_df.loc[
    threshold_df['f1'].idxmax()]
print('선택 임계값');
display(selected.to_frame('selected_value'))


,threshold,recall,precision,f1,predicted_churn_count
0,0.25,0.970,0.547,0.699,757
1,0.30,0.939,0.575,0.714,697
2,0.35,0.892,0.600,0.718,635
3,0.40,0.838,0.637,0.724,562
4,0.45,0.761,0.665,0.710,489
5,0.50,0.672,0.712,0.692,403
6,0.55,0.595,0.745,0.661,341
7,0.60,0.499,0.753,0.600,283
8,0.65,0.424,0.774,0.548,234
9,0.70,0.302,0.777,0.435,166


선택 임계값


,selected_value
threshold,0.400000
recall,0.838407
precision,0.637011
f1,0.723964
predicted_churn_count,562.000000


## 5. 결과 해석과 다음 단계
튜닝 전후 Validation 지표와 Train-Validation gap을 함께 확인한다. 최종 모델 합의 후 Train+Validation으로 재학습하고 Test는 한 번만 평가한다.